In [1]:
from api import maniskill
from api.video import show
import torch

In [2]:
# StackThree-v1 (custom task, api/maniskill_tasks.py): stack cubeB on cubeC, then cubeA on cubeB.
# Sparse reward only: +5 once cubeB is stacked on cubeC, +10 total once the full tower is complete.
# Open-loop random actions, so don't expect it to actually build the tower.
TASK = "StackThree-v1"
env = maniskill.make_env(TASK, num_envs=1, render=True)

obs, _ = env.reset(seed=0)
frames = [env.render()[0].cpu().numpy()]
for _ in range(env._max_episode_steps):
    action = torch.from_numpy(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render()[0].cpu().numpy())
env.close()

print("final reward:", reward.item(), "success:", info["success"].item())
show(frames, fps=env.unwrapped.control_freq)

/home/somatic/faraz/control-proj/control/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


final reward: 0.0 success: False


In [3]:
# A real demonstration, since there's no official dataset for a custom task: a motion-planned scripted
# solution (api.maniskill_tasks.solve_stack_three) that reasons over privileged simulator state (object
# poses, grasp geometry from the cube's oriented bounding box) rather than vision, grasps cubeB and places
# it on cubeC, then grasps cubeA and places it on cubeB. Recorded once for seed=0 via ManiSkill's own
# RecordEpisode wrapper, into <repo>/demos/<task>/motionplanning/ (api.maniskill.REPO_DEMO_DIR) -- committed
# to the repo (it's tiny, ~90KB) so other machines don't need to regenerate it. api.maniskill.load_demo
# checks that path first, before ManiSkill's own download cache.
import gymnasium as gym
from mani_skill.utils.wrappers.record import RecordEpisode

import api.maniskill_tasks as maniskill_tasks

if not (maniskill.REPO_DEMO_DIR / TASK / "motionplanning" / "trajectory.h5").exists():
    gen_env = gym.make(TASK, obs_mode="none", control_mode="pd_joint_pos", render_mode="rgb_array", sim_backend="auto")
    gen_env = RecordEpisode(
        gen_env,
        output_dir=str(maniskill.REPO_DEMO_DIR / TASK / "motionplanning"),
        trajectory_name="trajectory",
        save_video=False,
        source_type="motionplanning",
        source_desc="custom StackThree-v1 scripted solution",
        record_reward=False,
        save_on_reset=False,
    )
    res = maniskill_tasks.solve_stack_three(gen_env, seed=0)
    gen_env.flush_trajectory()
    gen_env.close()
    print("generated demo, recorded success:", res[-1]["success"].item())

In [4]:
# Open-loop replay of that demo, same mechanism as the StackCube-v1/PegInsertionSide-v1 replays in
# mani-skill-quickstart/rollout.ipynb: reset to the demo's exact recorded seed, then blindly replay its
# recorded actions.
demo = maniskill.load_demo(TASK)
print(f"{len(demo.actions)} actions, control_mode={demo.control_mode}, recorded success={demo.success}")

env = maniskill.make_env(
    TASK, num_envs=1, render=True,
    control_mode=demo.control_mode,
    max_episode_steps=len(demo.actions) + 1,
)
obs, _ = env.reset(**demo.reset_kwargs)
frames = [env.render()[0].cpu().numpy()]
for a in demo.actions:
    obs, reward, terminated, truncated, info = env.step(torch.from_numpy(a).unsqueeze(0))
    frames.append(env.render()[0].cpu().numpy())
env.close()

print("replay success:", info["success"].item())
show(frames, fps=env.unwrapped.control_freq)

234 actions, control_mode=pd_joint_pos, recorded success=True


replay success: True
